# 02 编码器模块 (core.encoders)

提供各类特征编码功能，所有编码器均遵循 sklearn Transformer 接口规范。

**支持的编码器：**
- `WOEEncoder`: 证据权重编码（风控专用）
- `TargetEncoder`: 目标编码
- `CountEncoder`: 计数编码
- `OneHotEncoder`: 独热编码
- `OrdinalEncoder`: 序数编码
- `QuantileEncoder`: 分位数编码
- `CatBoostEncoder`: CatBoost编码（有序目标编码）
- `GBMEncoder`: 梯度提升树编码器（叶子节点/Embedding）
- `CardinalityEncoder`: 高基数降维编码器

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [1]:
import os, sys
sys.path.append('../')

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from hscredit import init_setting
from hscredit.core.encoders import (
    WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder,
    OrdinalEncoder, QuantileEncoder, CatBoostEncoder,
    GBMEncoder, CardinalityEncoder
)
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import ks, auc

init_setting()

df = pd.read_excel('hscredit_yyp.xlsx')
df['target'] = (df['MOB1'] > 3).astype(int)

numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '轻花老客海纳子分V1']

# 先分箱，再进行编码
X_raw = df[numeric_features].fillna(df[numeric_features].median())

binner = OptimalBinning(method='quantile', max_n_bins=5, min_bin_size=0.05)
df_binned = binner.fit_transform(X_raw, df['target']).astype(str)
df_binned.columns = [f'{c}_bin' for c in df_binned.columns]

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")
print(f"分箱后列: {df_binned.columns.tolist()}")
display(df_binned.head())

样本数: 970
坏样本率: 16.70%
分箱后列: ['中智小牛分C3_bin', '珊瑚92_bin', '极光欺诈分6v1_bin', '轻花老客海纳子分V1_bin']


,中智小牛分C3_bin,珊瑚92_bin,极光欺诈分6v1_bin,轻花老客海纳子分V1_bin
0,1,1,0,1
1,1,1,0,1
2,1,1,0,1
3,1,1,0,1
4,1,1,0,1


## 1. WOEEncoder - 证据权重编码（风控专用）

In [3]:
# WOE编码
woe_encoder = WOEEncoder()
woe_encoder.fit(df_binned, df['target'])
df_woe = woe_encoder.transform(df_binned.copy())

print("WOE编码结果预览:")
display(df_woe.head())

print("\nWOE映射表:")
for col in df_woe.columns:
    print(f"\n{col}:")
    mapping = woe_encoder.get_mapping(col)
    for k, v in mapping.items():
        print(f"  {k}: {v:.4f}")

WOE编码结果预览:


,中智小牛分C3_bin,珊瑚92_bin,极光欺诈分6v1_bin,轻花老客海纳子分V1_bin
0,0.0573,0.0188,-0.0388,-0.2970
1,0.0573,0.0188,-0.0388,-0.2970
2,0.0573,0.0188,-0.0388,-0.2970
3,0.0573,0.0188,-0.0388,-0.2970
4,0.0573,0.0188,-0.0388,-0.2970



WOE映射表:

中智小牛分C3_bin:
  1: 0.0573
  0: 0.2190
  2: -0.5851
  nan: 0.0000
  __UNKNOWN__: 0.0000

珊瑚92_bin:
  1: 0.0188
  0: -0.1256
  nan: 0.0000
  __UNKNOWN__: 0.0000

极光欺诈分6v1_bin:
  0: -0.0388
  1: 0.1943
  nan: 0.0000
  __UNKNOWN__: 0.0000

轻花老客海纳子分V1_bin:
  1: -0.2970
  2: 0.4501
  0: -0.6345
  4: 1.1589
  3: 0.2277
  nan: 0.0000
  __UNKNOWN__: 0.0000


## 2. TargetEncoder - 目标编码

In [4]:
# Target编码
target_encoder = TargetEncoder(
    cols=['中智小牛分C3_bin'],
    smoothing=10
)
target_encoder.fit(df_binned[['中智小牛分C3_bin']], df['target'])

df_target = df_binned[['中智小牛分C3_bin']].copy()
df_target['target_encoded'] = target_encoder.transform(df_target)['中智小牛分C3_bin']

print("Target编码结果预览:")
display(df_target.head(10))

Target编码结果预览:


,中智小牛分C3_bin,target_encoded
0,1,0.1755
1,1,0.1755
2,1,0.1755
3,1,0.1755
4,1,0.1755
5,1,0.1755
6,1,0.1755
7,1,0.1755
8,1,0.1755
9,1,0.1755


## 3. CountEncoder - 计数编码

In [5]:
# Count编码
count_encoder = CountEncoder(
    cols=['中智小牛分C3_bin'],
    normalize=False
)
count_encoder.fit(df_binned[['中智小牛分C3_bin']])

df_count = df_binned[['中智小牛分C3_bin']].copy()
df_count['count_encoded'] = count_encoder.transform(df_count)['中智小牛分C3_bin']

print("Count编码结果预览:")
display(df_count.head(10))

Count编码结果预览:


,中智小牛分C3_bin,count_encoded
0,1,672
1,1,672
2,1,672
3,1,672
4,1,672
5,1,672
6,1,672
7,1,672
8,1,672
9,1,672


## 4. OneHotEncoder - 独热编码

In [6]:
# OneHot编码
onehot_encoder = OneHotEncoder(
    cols=['中智小牛分C3_bin'],
    drop='first'
)
onehot_encoder.fit(df_binned[['中智小牛分C3_bin']], df['target'])

df_onehot = onehot_encoder.transform(df_binned[['中智小牛分C3_bin']].copy())

print(f"OneHot编码后特征数: {df_onehot.shape[1]}")
print("OneHot编码结果预览:")
display(df_onehot.head(10))

OneHot编码后特征数: 2
OneHot编码结果预览:


,中智小牛分C3_bin_1,中智小牛分C3_bin_2
0,1,0
1,1,0
2,1,0
3,1,0
4,1,0
5,1,0
6,1,0
7,1,0
8,1,0
9,1,0


## 5. OrdinalEncoder - 序数编码

In [7]:
# Ordinal编码 - 按指定顺序映射
ordinal_encoder = OrdinalEncoder(
    cols=['中智小牛分C3_bin'],
    mapping={'中智小牛分C3_bin': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}}
)
ordinal_encoder.fit(df_binned[['中智小牛分C3_bin']])

df_ordinal = df_binned[['中智小牛分C3_bin']].copy()
df_ordinal['ordinal_encoded'] = ordinal_encoder.transform(df_ordinal)['中智小牛分C3_bin']

print("Ordinal编码结果预览:")
display(df_ordinal.head(10))

Ordinal编码结果预览:


,中智小牛分C3_bin,ordinal_encoded
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1
5,1,1
6,1,1
7,1,1
8,1,1
9,1,1


## 6. QuantileEncoder - 分位数编码

In [8]:
# Quantile编码 - 按分位数（中位数）编码
quantile_encoder = QuantileEncoder(
    cols=['中智小牛分C3_bin'],
    quantile=0.5  # 中位数
)
quantile_encoder.fit(df_binned[['中智小牛分C3_bin']], df['target'])

df_quantile = df_binned[['中智小牛分C3_bin']].copy()
df_quantile['quantile_encoded'] = quantile_encoder.transform(df_quantile)['中智小牛分C3_bin']

print("Quantile编码结果预览:")
display(df_quantile.head(10))

Quantile编码结果预览:


,中智小牛分C3_bin,quantile_encoded
0,1,0.0000
1,1,0.0000
2,1,0.0000
3,1,0.0000
4,1,0.0000
5,1,0.0000
6,1,0.0000
7,1,0.0000
8,1,0.0000
9,1,0.0000


## 7. CatBoostEncoder - CatBoost有序目标编码

In [9]:
# CatBoost编码 - 有序目标编码，减少过拟合
catboost_encoder = CatBoostEncoder(
    cols=['中智小牛分C3_bin'],
    sigma=0.1,  # 添加高斯噪声防止过拟合
    random_state=42
)
catboost_encoder.fit(df_binned[['中智小牛分C3_bin']], df['target'])

df_catboost = df_binned[['中智小牛分C3_bin']].copy()
df_catboost['catboost_encoded'] = catboost_encoder.transform(df_catboost)['中智小牛分C3_bin']

print("CatBoost编码结果预览:")
display(df_catboost.head(10))

CatBoost编码结果预览:


,中智小牛分C3_bin,catboost_encoded
0,1,0.1756
1,1,0.1756
2,1,0.1756
3,1,0.1756
4,1,0.1756
5,1,0.1756
6,1,0.1756
7,1,0.1756
8,1,0.1756
9,1,0.1756


## 8. GBMEncoder - 梯度提升树编码器

In [10]:
# GBM编码 - 使用梯度提升树提取叶子节点或Embedding
X_for_gbm = df[numeric_features].fillna(df[numeric_features].median())
y_for_gbm = df['target']

X_train_gbm, X_test_gbm, y_train_gbm, y_test_gbm = train_test_split(
    X_for_gbm, y_for_gbm, test_size=0.3, random_state=42
)

# 叶子节点编码
gbm_leaves = GBMEncoder(
    model_type='lightgbm',
    n_estimators=50,
    max_depth=3,
    output_type='leaves',  # 输出叶子节点编码
    random_state=42
)
gbm_leaves.fit(X_train_gbm, y_train_gbm)
leaves_train = gbm_leaves.transform(X_train_gbm)

print("GBM叶子节点编码结果预览:")
print(f"形状: {leaves_train.shape}")
display(pd.DataFrame(leaves_train[:5], columns=[f'leaf_{i}' for i in range(leaves_train.shape[1])]))

GBM叶子节点编码结果预览:
形状: (679, 50)


,leaf_0,leaf_1,leaf_2,leaf_3,leaf_4,leaf_5,leaf_6,leaf_7,leaf_8,leaf_9,...,leaf_40,leaf_41,leaf_42,leaf_43,leaf_44,leaf_45,leaf_46,leaf_47,leaf_48,leaf_49
481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
944,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. CardinalityEncoder - 高基数降维编码器

In [11]:
# Cardinality编码 - 高基数类别降维
# 先创建一些高基数类别特征
np.random.seed(42)
df['category_1'] = pd.cut(df['中智小牛分C3'].fillna(df['中智小牛分C3'].median()), bins=20).astype(str)
df['category_2'] = pd.cut(df['珊瑚92'].fillna(df['珊瑚92'].median()), bins=15).astype(str)

card_encoder = CardinalityEncoder(
    cols=['category_1', 'category_2'],
    max_categories=5,
)
card_encoder.fit(df[['category_1', 'category_2']])

df_card = card_encoder.transform(df[['category_1', 'category_2']].copy())

print("Cardinality编码结果预览:")
display(df_card.head(10))

print("\n各类别的频次:")
for col in df_card.columns:
    print(f"\n{col}:")
    print(df_card[col].value_counts().head())

Cardinality编码结果预览:


,category_1,category_2
0,"(618.4, 637.7]","(604.0, 631.333]"
1,"(618.4, 637.7]","(604.0, 631.333]"
2,"(618.4, 637.7]","(604.0, 631.333]"
3,"(618.4, 637.7]","(604.0, 631.333]"
4,"(618.4, 637.7]","(604.0, 631.333]"
5,"(618.4, 637.7]","(604.0, 631.333]"
6,"(618.4, 637.7]","(604.0, 631.333]"
7,"(618.4, 637.7]","(604.0, 631.333]"
8,"(618.4, 637.7]","(604.0, 631.333]"
9,"(618.4, 637.7]","(604.0, 631.333]"



各类别的频次:

category_1:
category_1
(618.4, 637.7]    681
other             214
(579.8, 599.1]     27
(599.1, 618.4]     24
(695.6, 714.9]     24
Name: count, dtype: int64

category_2:
category_2
(604.0, 631.333]      776
other                  91
(576.667, 604.0]       50
(631.333, 658.667]     33
(658.667, 686.0]       20
Name: count, dtype: int64


## 10. Pipeline中使用编码器

In [12]:
from sklearn.pipeline import Pipeline
from hscredit.core.models import LogisticRegression

# 准备数据
feature_cols = ['中智小牛分C3_bin', '珊瑚92_bin', '极光欺诈分6v1_bin', '轻花老客海纳子分V1_bin']
X = df_binned[feature_cols].copy()
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 对比不同编码器在LR模型上的效果
encoders = {
    'WOE': WOEEncoder(),
    'Target': TargetEncoder(cols=feature_cols, smoothing=10),
    'Count': CountEncoder(cols=feature_cols),
}

results = []
for name, encoder in encoders.items():
    X_train_enc = encoder.fit_transform(X_train, y_train)
    X_test_enc = encoder.transform(X_test)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_enc, y_train)
    y_prob = model.predict_proba(X_test_enc)[:, 1]
    
    results.append({
        '编码器': name,
        'KS': ks(y_test, y_prob),
        'AUC': auc(y_test, y_prob),
    })

print("=== 编码器效果对比 ===")
display(pd.DataFrame(results))

=== 编码器效果对比 ===


,编码器,KS,AUC
0,WOE,0.2450,0.6202
1,Target,0.2780,0.6539
2,Count,0.1154,0.5385
